<a href="https://colab.research.google.com/github/MarcosRigal/AP/blob/main/adapted_performance_metrics_for_oc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Síntesis del Paper: Métricas Ordinales adaptadas a Intervalos

En este cuaderno se explica la adaptación de las métricas para **clasificación ordinal** cuando las clases se definen como **intervalos** de longitudes potencialmente desiguales. Además, se ilustra cómo visualizar matrices de confusión y cómo calcular las métricas *MAE*, *MAE_int*, *TC* y *TC_int*.

## 1. Función para graficar la Matriz de Confusión

Dibujamos la matriz de confusión como un *heatmap*, para así visualizar fácilmente los aciertos y errores del clasificador.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(cm, classes, title='Matriz de Confusión', normalize=False):
    """
    Dibuja un mapa de calor con la matriz de confusión.
    - cm: Matriz de confusión (numpy array) de tamaño [r, r].
    - classes: Lista o tupla con los nombres (etiquetas) de las clases.
    - title: Título de la figura.
    - normalize: Si es True, normaliza la matriz por filas.
    """
    plt.figure()
    if normalize:
        cm_sum = cm.sum(axis=1, keepdims=True)
        cm_sum[cm_sum == 0] = 1  # Evita división por cero en filas con suma=0
        cm = cm / cm_sum

    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], fmt),
                     ha="center", va="center", color="black")

    plt.ylabel('Clase Predicha')
    plt.xlabel('Clase Real')
    plt.tight_layout()
    plt.show()

: 

## 2. Funciones para métricas Ordinales y Adaptadas a Intervalos

A continuación, definimos:
- **MAE (Mean Absolute Error)** en su versión ordinal.
- **TC (Total Misclassification Cost)** en su versión ordinal.
- **MAE_int** y **TC_int**, sus versiones adaptadas a intervalos, usando la "distancia entre intervalos" (ejemplo: distancia de Hausdorff) y la densidad (número de instancias dividido entre la longitud del intervalo).

Además, incluimos funciones auxiliares para:
- Calcular la distancia entre intervalos (`dist_interval`).
- Normalizar una métrica dividiéndola por su valor máximo posible.

In [ ]:
def mae_ordinal(cm):
    """
    MAE (Mean Absolute Error) para clasificación ordinal.
    Asume que las clases están indexadas 0,1,...,r-1
    y la distancia entre clases es |i-j|.
    """
    N = cm.sum()
    r = cm.shape[0]
    total_error = 0.0
    for i in range(r):
        for j in range(r):
            total_error += cm[i,j] * abs(i-j)
    return total_error / N if N > 0 else 0

def tc_ordinal(cm, n):
    """
    TC (Total Misclassification Cost) versión ordinal.
    - cm: matriz de confusión.
    - n: array con la cantidad total de instancias verdaderas en cada clase.
    El coste = ((N - n[j]) / n[i]) * |i-j| para i!=j.
    """
    r = cm.shape[0]
    N = cm.sum()
    total_cost = 0.0
    for i in range(r):
        for j in range(r):
            if i != j:
                cost_ij = ( (N - n[j]) / n[i] ) * abs(i-j)
                total_cost += cm[i,j] * cost_ij
    return total_cost

def dist_interval(i, j, lengths):
    """
    Función de ejemplo para la distancia entre intervalos I_i e I_j.
    Suponemos que:
       I_0 = [0, lengths[0]), I_1 = [lengths[0], lengths[0]+lengths[1]), etc.
    Simplificamos usando la suma de longitudes intermedias.
    """
    if i == j:
        return 0.0
    if i > j:
        i, j = j, i
    return sum(lengths[i+1:j+1])

def mae_interval(cm, lengths):
    """
    MAE_int: versión de MAE que usa la distancia real entre intervalos.
    """
    N = cm.sum()
    r = cm.shape[0]
    total_error = 0.0
    for i in range(r):
        for j in range(r):
            d = dist_interval(i, j, lengths)
            total_error += cm[i,j] * d
    return total_error / N if N > 0 else 0

def tc_interval(cm, n, lengths):
    """
    TC_int: versión de TC adaptada a intervalos.
    Reemplaza n_j por densidad (n_j / lengths[j]) y |i-j| por dist_interval(i,j).
    """
    N = cm.sum()
    r = cm.shape[0]
    total_cost = 0.0

    dens = np.array([n[j] / lengths[j] for j in range(r)])  # densidad en cada intervalo

    for i in range(r):
        for j in range(r):
            if i != j:
                cost_ij = (dens.sum() - dens[j]) / dens[i]
                d = dist_interval(i, j, lengths)
                total_cost += cm[i,j] * cost_ij * d
    return total_cost

def normalize_metric(value, max_value):
    """
    Normaliza una métrica para que sea un valor en [0,1],
    dividiéndola entre su valor máximo possible (max_value).
    """
    if max_value == 0:
        return 0.0
    return value / max_value

## 3. Ejemplo de uso

Vamos a crear matrices de confusión de ejemplo y comparar las métricas *ordinales* contra las métricas *interval-scale*.
También pintaremos la matriz de confusión para ver su distribución de aciertos/errores.

In [ ]:
# Ejemplo con 3 clases-intervalos
# Supongamos que cada intervalo tiene esta longitud:
lengths = [1.0, 0.4, 2.0]  # I1=[0,1), I2=[1,1.4), I3=[1.4,3.4)

# Matriz de confusión de ejemplo
cm_example = np.array([[3, 2, 1],
                       [2, 2, 2],
                       [0, 1, 2]])

# Calculamos la cantidad de instancias reales en cada clase (suma por columnas)
n_example = cm_example.sum(axis=0)

print("Matriz de Confusión (sin normalizar):\n", cm_example)
plot_confusion_matrix(cm_example, classes=["I1","I2","I3"],
                      title="Matriz de Confusión - Sin Normalizar", normalize=False)

plot_confusion_matrix(cm_example, classes=["I1","I2","I3"],
                      title="Matriz de Confusión - Normalizada (filas)", normalize=True)

# Métricas Ordinales
mae_ord = mae_ordinal(cm_example)
tc_ord = tc_ordinal(cm_example, n_example)
print(f"MAE (ordinal) = {mae_ord:.4f}")
print(f"TC (ordinal)  = {tc_ord:.4f}")

# Métricas Interval-Scale
mae_int_val = mae_interval(cm_example, lengths)
tc_int_val = tc_interval(cm_example, n_example, lengths)
print(f"MAE_int = {mae_int_val:.4f}")
print(f"TC_int  = {tc_int_val:.4f}")